# Bitcoin Miner Efficiency & Obsolescence Dashboard

This interactive data application implements a reverse-engineered economic simulation for Bitcoin mining. Instead of analyzing profitability by fixed hardware brand names, this simulator isolates **thermodynamic efficiency ($J/\text{TH}$)** as the ultimate dependent variable driving operational survival.

## Project Framework
* **Macro Inputs:** Real-time controls for Bitcoin Spot Price, Global Network Difficulty, Mining Pool Fees, and Reward Eras (Pre vs. Post 2028 Halving).
* **Operational Variable:** Grid Electricity Tariff ($/kWh).
* **Primary Output Engine:** Computes the absolute Maximum Viable Efficiency Cap alongside an optimized Industrial Sourcing Target.
* **Core Visualization:** Generates a continuous mathematical yield curve scaled to **Petahashes per second (PH/s)**, illustrating the operational boundary between the "Profit Zone" and the "Obsolescence Zone."

### The Mathematical Engine
1. **Maximum Viable Efficiency ($J/\text{TH}$):**
   $$E_{\text{max}} = \left( \frac{\left( \frac{10^{12} \times 86400 \times BR}{Difficulty \times 2^{32}} \right) \times P_{\text{BTC}} \times (1 - Fee)}{R_{\text{elec}}} \right) \times \left( \frac{1000}{24} \right)$$

2. **Daily Net Profit per PH/s ($P_{\text{net}}$):**
   $$P_{\text{net}} = \left( R_{\text{TH}} \times 1000 \right) - \left( \frac{E}{1000} \times 1000 \times 24 \times R_{\text{elec}} \right)$$

In [2]:
# Step 1: Initialize environment and manage package dependencies
!pip install dash plotly pandas


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Interactive Simulation & UI Layout Engine

The code cell below constructs the interactive data pipeline. It leverages the modern `app.run()` syntax optimized for Jupyter Notebook environments, binding a unified CSS container interface to your localized analytical equations.

In [3]:
import dash
from dash import dcc, html, Input, Output
import plotly.graph_objects as go
import pandas as pd

# Initialize the modern Dash app instance
app = dash.Dash(__name__)

# Define clean CSS layout structures inline for portable deployment
app.layout = html.Div([
    # Executive Header
    html.Div([
        html.H1("Bitcoin Miner Efficiency & Obsolescence Simulator", 
                style={'margin': '0', 'fontFamily': 'Segoe UI, Arial', 'color': '#2c3e50', 'fontWeight': '600'}),
        html.P("Reverse-engineered industrial hardware valuation model for the 2028 Halving Era.", 
               style={'margin': '5px 0 0 0', 'color': '#7f8c8d', 'fontFamily': 'Segoe UI, Arial'})
    ], style={'padding': '20px', 'backgroundColor': '#f8f9fa', 'borderBottom': '1px solid #e2e8f0'}),
    
    # Master Workspace Container (Flexbox layout)
    html.Div([
        
        # 🎛️ ZONE A: THE INPUT CONSOLE (SIDEBAR - 30% Width)
        html.Div([
            html.H3("Market & Macro Drivers", style={'fontFamily': 'Segoe UI', 'color': '#34495e', 'marginTop': '0'}),
            
            html.Label("Bitcoin Spot Price Target ($ USD):", style={'fontWeight': 'bold', 'display': 'block'}),
            dcc.Slider(id='btc-price', min=60000, max=200000, step=5000, value=120000,
                       marks={i: f"${i:,}" for i in range(60000, 210000, 40000)}),
            
            html.Label("Reward Era Selection:", style={'fontWeight': 'bold', 'display': 'block', 'marginTop': '20px'}),
            dcc.Dropdown(id='reward-era',
                         options=[
                             {'label': 'Current Era (3.125 BTC)', 'value': 'pre'},
                             {'label': '2028 Post-Halving Era (1.5625 BTC)', 'value': 'post'}
                         ], value='post', clearable=False, style={'marginBottom': '20px'}),
            
            html.Label("Network Difficulty Baseline (T):", style={'fontWeight': 'bold', 'display': 'block'}),
            dcc.Slider(id='difficulty', min=100, max=300, step=10, value=165,
                       marks={i: f"{i}T" for i in range(100, 310, 50)}),
            
            html.Hr(style={'margin': '25px 0', 'border': '0', 'borderTop': '1px solid #dcdde1'}),
            
            html.H3("Operational Parameters", style={'fontFamily': 'Segoe UI', 'color': '#34495e'}),
            
            html.Label("Grid Electricity Tariff ($/kWh):", style={'fontWeight': 'bold', 'display': 'block'}),
            dcc.Slider(id='elec-cost', min=0.02, max=0.15, step=0.005, value=0.05,
                       marks={i: f"${i:.2f}" for i in [0.02, 0.05, 0.08, 0.11, 0.15]}),
            
            html.Label("Mining Pool Fees (%):", style={'fontWeight': 'bold', 'display': 'block', 'marginTop': '20px'}),
            dcc.Slider(id='pool-fee', min=0.0, max=0.05, step=0.005, value=0.02,
                       marks={i: f"{i*100:.1f}%" for i in [0.0, 0.02, 0.05]}),
                         
        ], style={
            'width': '28%', 'padding': '25px', 'backgroundColor': '#ffffff', 
            'boxShadow': '2px 0 5px rgba(0,0,0,0.05)', 'minHeight': '80vh', 'boxSizing': 'border-box'
        }),
        
        # RIGHT CANVAS (70% Width)
        html.Div([
            
            # 📊 ZONE B: EXECUTIVE SUMMARY CARDS (Top Right)
            html.Div(id='kpi-cards-container', style={
                'display': 'flex', 'justifyContent': 'space-between', 'marginBottom': '25px'
            }),
            
            # 📉 ZONE C: THE EFFICIENCY SPECTRUM CURVE (Bottom Right)
            html.Div([
                dcc.Graph(id='efficiency-curve-graph')
            ], style={'backgroundColor': '#ffffff', 'padding': '15px', 'borderRadius': '8px', 'boxShadow': '0 4px 6px rgba(0,0,0,0.02)'})
            
        ], style={'width': '72%', 'padding': '25px', 'boxSizing': 'border-box'})
        
    ], style={'display': 'flex', 'flexDirection': 'row', 'backgroundColor': '#f1f2f6'})
])

# Master call-back interface linking inputs to Zone B and Zone C outputs simultaneously
@app.callback(
    [Output('kpi-cards-container', 'children'),
     Output('efficiency-curve-graph', 'figure')],
    [Input('btc-price', 'value'),
     Input('reward-era', 'value'),
     Input('difficulty', 'value'),
     Input('elec-cost', 'value'),
     Input('pool-fee', 'value')]
)
def execute_simulation_pipeline(btc_price, era_selection, diff_t, elec_cost, pool_fee):
    # 1. Variable Pre-Processing
    block_reward = 3.125 if era_selection == 'pre' else 1.5625
    difficulty_raw = diff_t * 1_000_000_000_000 # Convert T label to raw numerical difficulty
    
    # 2. Compute Master Target Macro Outputs
    # Revenue factor per 1 TH/s per day
    raw_th_revenue_day = ((10**12 * 86400 * block_reward) / (difficulty_raw * 2**32)) * btc_price
    net_th_revenue_day = raw_th_revenue_day * (1 - pool_fee)
    
    # Reverse-engineered Maximum Viable Efficiency Cap (Breakeven Line)
    max_viable_efficiency = (net_th_revenue_day / elec_cost) * (1000.0 / 24.0)
    
    # Ideal Sourcing Target (30% Profit Cushion Margin Strategy)
    recommended_target = max_viable_efficiency * 0.70
    
    # Evaluate Market Safety Context
    if max_viable_efficiency < 12.0:
        safety_text = "CRITICAL OBSOLESCENCE RISK"
        safety_color = "#e74c3c"
        safety_sub = "Only unreleased/next-gen hydro fleets viable."
    elif max_viable_efficiency < 17.0:
        safety_text = "HIGH RISK / TIGHT MARGINS"
        safety_color = "#f39c12"
        safety_sub = "Requires ultra-efficient 3nm air or hydro setups."
    else:
        safety_text = "HEALTHY REVENUE RUNWAY"
        safety_color = "#2ecc71"
        safety_sub = "Standard commercial hardware remains operational."

    # 3. Generate Zone B Layout (KPI Cards Structure)
    kpi_layouts = [
        html.Div([
            html.Div("MAX VIABLE EFFICIENCY", style={'fontSize': '12px', 'fontWeight': 'bold', 'color': '#7f8c8d'}),
            html.Div(f"{max_viable_efficiency:.2f} J/TH", style={'fontSize': '26px', 'fontWeight': 'bold', 'color': '#c0392b'}),
            html.Div("Absolute Breakeven Ceiling", style={'fontSize': '11px', 'color': '#95a5a6', 'marginTop': '4px'})
        ], style={'backgroundColor': '#fff', 'padding': '20px', 'borderRadius': '6px', 'width': '31%', 'boxShadow': '0 2px 5px rgba(0,0,0,0.05)', 'borderLeft': '5px solid #c0392b'}),
        
        html.Div([
            html.Div("RECOMMENDED TARGET", style={'fontSize': '12px', 'fontWeight': 'bold', 'color': '#7f8c8d'}),
            html.Div(f"{recommended_target:.2f} J/TH", style={'fontSize': '26px', 'fontWeight': 'bold', 'color': '#27ae60'}),
            html.Div("Builds a 30% OpEx Cushion", style={'fontSize': '11px', 'color': '#95a5a6', 'marginTop': '4px'})
        ], style={'backgroundColor': '#fff', 'padding': '20px', 'borderRadius': '6px', 'width': '31%', 'boxShadow': '0 2px 5px rgba(0,0,0,0.05)', 'borderLeft': '5px solid #27ae60'}),
        
        html.Div([
            html.Div("FLEET HEALTH STATUS", style={'fontSize': '12px', 'fontWeight': 'bold', 'color': '#7f8c8d'}),
            html.Div(safety_text, style={'fontSize': '16px', 'fontWeight': 'bold', 'color': safety_color, 'marginTop': '8px'}),
            html.Div(safety_sub, style={'fontSize': '11px', 'color': '#95a5a6', 'marginTop': '8px'})
        ], style={'backgroundColor': '#fff', 'padding': '20px', 'borderRadius': '6px', 'width': '31%', 'boxShadow': '0 2px 5px rgba(0,0,0,0.05)', 'borderLeft': f'5px solid {safety_color}'})
    ]

    # 4. Generate Zone C Data Matrix (Continuous Curve Calculations)
    efficiency_spectrum = list(range(8, 33)) # From premium 8 J/TH to legacy 32 J/TH
    ph_profits = []
    
    for eff in efficiency_spectrum:
        daily_power_cost_ph = (eff / 1000.0) * 1000 * 24 * elec_cost
        daily_rev_ph = net_th_revenue_day * 1000
        net_profit_ph = daily_rev_ph - daily_power_cost_ph
        ph_profits.append(net_profit_ph)

    # Build Plotly Data Structure
    fig = go.Figure()
    
    # Draw Continuous Efficiency Curve Trace
    fig.add_trace(go.Scatter(
        x=efficiency_spectrum, y=ph_profits,
        mode='lines+markers', name='Daily Profit per PH/s',
        line=dict(color='#2980b9', width=3),
        marker=dict(size=6)
    ))
    
    # Draw Master Vertical Breakeven Line Threshold
    fig.add_vline(x=max_viable_efficiency, line_width=2, line_dash="dash", line_color="#c0392b",
                  annotation_text=f"Breakeven Line ({max_viable_efficiency:.1f} J/TH)", annotation_position="top right")
    
    # Apply Visual Background Zoning (Green vs Red Background Blocks)
    fig.add_vrect(x0=8, x1=min(max(max_viable_efficiency, 8), 32), fillcolor="#2ecc71", opacity=0.08, line_width=0, layer="below")
    fig.add_vrect(x0=min(max(max_viable_efficiency, 8), 32), x1=32, fillcolor="#e74c3c", opacity=0.08, line_width=0, layer="below")

    fig.update_layout(
        title="Daily Operational Net Profit / Loss Curve (Per PH/s)",
        xaxis_title="ASIC Thermodynamic Hardware Efficiency Profile (J/TH) — Lower is More Efficient",
        yaxis_title="Net Yield Balance ($ USD / Day)",
        template="plotly_white",
        xaxis=dict(range=[8, 32], dtick=2),
        yaxis=dict(zeroline=True, zerolinewidth=1.5, zerolinecolor='#7f8c8d'),
        margin=dict(l=40, r=40, t=50, b=40)
    )

    return kpi_layouts, fig

# Launch application server directly inline within the Jupyter session
if __name__ == '__main__':
    app.run(debug=True, jupyter_mode="inline", port=8052)

## Dynamic Procurement & Hardware Sourcing Engine

The following implementation introduces a fixed array of real-world ASIC mining hardware models (containing their manufacturer hashrate and raw wall-power metrics). 

When the macroeconomic parameters shift above via the dashboard sliders, the engine automatically maps the calculated target thresholds directly onto these live hardware configurations. The resulting table dynamically computes each specific machine's actual daily USD net cash flow and tags its viability status automatically.

## Bitcoin Enterprise Procurement & Payback Dashboard

This production notebook deployment implements a self-contained real-time optimization loop for Bitcoin hardware infrastructure. It pairs the thermodynamic efficiency threshold logic from our spreadsheet models with a dynamic commercial hardware index. 

By integrating wholesale equipment retail spot-pricing variables, the grid automatically calculates operational yields alongside the **CapEx Payback Window (Days to ROI)** for specific physical mining units deployed under these exact macro conditions.

In [6]:
import dash
from dash import dcc, html, Input, Output, dash_table
import plotly.graph_objects as go
import pandas as pd

# 1. BITCOIN SPECIFIC HARDWARE INDEX WITH WHOLESALE MARKUP VALUE PARAMETERS
# This array holds standard manufacturing baselines and live capital market pricing nodes.
BTC_ASIC_CATALOG = [
    {"Model": "Antminer S23 Hydro (Next-Gen)", "Hashrate_TH": 580.0, "Power_W": 5510.0, "Efficiency_JTH": 9.5, "Market_Price_USD": 14500.0},
    {"Model": "Antminer S21 XP Hydro", "Hashrate_TH": 473.0, "Power_W": 5676.0, "Efficiency_JTH": 12.0, "Market_Price_USD": 9460.0},
    {"Model": "Whatsminer M63S+", "Hashrate_TH": 390.0, "Power_W": 5070.0, "Efficiency_JTH": 13.0, "Market_Price_USD": 6240.0},
    {"Model": "Antminer S21 Pro", "Hashrate_TH": 234.0, "Power_W": 3322.0, "Efficiency_JTH": 14.2, "Market_Price_USD": 3270.0},
    {"Model": "Antminer S21 Standard", "Hashrate_TH": 200.0, "Power_W": 3500.0, "Efficiency_JTH": 17.5, "Market_Price_USD": 2400.0},
    {"Model": "Antminer T21", "Hashrate_TH": 190.0, "Power_W": 3610.0, "Efficiency_JTH": 19.0, "Market_Price_USD": 2185.0},
    {"Model": "Antminer S19 XP Legacy", "Hashrate_TH": 141.0, "Power_W": 3032.0, "Efficiency_JTH": 21.5, "Market_Price_USD": 1120.0}
]

df_hardware = pd.DataFrame(BTC_ASIC_CATALOG)

# Initialize application canvas
app = dash.Dash(__name__)

app.layout = html.Div([
    # Enterprise Visual Branding Header
    html.Div([
        html.H1("Bitcoin Infrastructure Efficiency & Payback Engine", 
                style={'margin': '0', 'fontFamily': 'Segoe UI, Arial', 'color': '#2c3e50', 'fontWeight': '600'}),
        html.P("Reverse-engineered ASIC sourcing model driven by network economics and wholesale asset price layers.", 
               style={'margin': '5px 0 0 0', 'color': '#7f8c8d', 'fontFamily': 'Segoe UI, Arial'})
    ], style={'padding': '20px', 'backgroundColor': '#f8f9fa', 'borderBottom': '1px solid #e2e8f0'}),
    
    # Flexbox Workspace Container
    html.Div([
        
        # 🎛️ ZONE A: THE BITCOIN INPUT PANEL (Sidebar - 30% Width)
        html.Div([
            html.H3("Macro Market Drivers", style={'fontFamily': 'Segoe UI', 'color': '#2c3e50', 'marginTop': '0'}),
            
            html.Label("Target Bitcoin Price ($ USD):", style={'fontWeight': 'bold', 'display': 'block'}),
            dcc.Slider(id='btc-price-slider', min=60000, max=200000, step=5000, value=120000,
                       marks={i: f"${i:,}" for i in range(60000, 210000, 40000)}),
            
            html.Label("Network Difficulty Baseline (T):", style={'fontWeight': 'bold', 'display': 'block', 'marginTop': '20px'}),
            dcc.Slider(id='network-difficulty-slider', min=100, max=300, step=10, value=165,
                       marks={i: f"{i}T" for i in range(100, 310, 50)}),
            
            html.Label("Reward Era Selection:", style={'fontWeight': 'bold', 'display': 'block', 'marginTop': '20px'}),
            dcc.Dropdown(id='halving-era-dropdown',
                         options=[
                             {'label': 'Current Era Baseline (3.125 BTC)', 'value': 'pre'},
                             {'label': '2028 Post-Halving Epoch (1.5625 BTC)', 'value': 'post'}
                         ], value='post', clearable=False, style={'marginBottom': '20px'}),
            
            html.Hr(style={'margin': '25px 0', 'border': '0', 'borderTop': '1px solid #dcdde1'}),
            
            html.H3("Grid & Facility Expenses", style={'fontFamily': 'Segoe UI', 'color': '#34495e'}),
            html.Label("Site Electricity Tariff ($/kWh):", style={'fontWeight': 'bold', 'display': 'block'}),
            dcc.Slider(id='grid-electricity-cost', min=0.02, max=0.15, step=0.005, value=0.05,
                       marks={i: f"${i:.2f}" for i in [0.02, 0.05, 0.08, 0.11, 0.15]}),
            
            html.Label("Pool Settlement Fees (%):", style={'fontWeight': 'bold', 'display': 'block', 'marginTop': '20px'}),
            dcc.Slider(id='pool-settlement-fee', min=0.0, max=0.05, step=0.005, value=0.02,
                       marks={i: f"{i*100:.1f}%" for i in [0.0, 0.02, 0.05]}),
                         
        ], style={
            'width': '28%', 'padding': '25px', 'backgroundColor': '#ffffff', 
            'boxShadow': '2px 0 5px rgba(0,0,0,0.05)', 'minHeight': '100vh', 'boxSizing': 'border-box'
        }),
        
        # RIGHT REPORTING CANVAS (70% Width)
        html.Div([
            
            # 📊 ZONE B: STRATEGIC READOUT CARDS
            html.Div(id='kpi-cards-grid', style={
                'display': 'flex', 'justifyContent': 'space-between', 'marginBottom': '25px'
            }),
            
            # 📉 ZONE C: THE ECONOMIC BOUNDARY PLOT
            html.Div([
                dcc.Graph(id='continuous-economic-curve')
            ], style={'backgroundColor': '#ffffff', 'padding': '15px', 'borderRadius': '8px', 'boxShadow': '0 4px 6px rgba(0,0,0,0.02)', 'marginBottom': '25px'}),
            
            # 📋 ZONE D: THE PROCUREMENT & PAYBACK MATRIX
            html.Div([
                html.H3("Dynamic ASIC Procurement & Payback Matrix", style={'fontFamily': 'Segoe UI', 'color': '#2c3e50', 'marginTop': '0'}),
                html.P("Real-time unit profit calculations and capital return schedules scaled to individual physical miner units.", style={'fontSize': '13px', 'color': '#7f8c8d'}),
                dash_table.DataTable(
                    id='realtime-procurement-table',
                    columns=[
                        {"name": "Hardware Model Name", "id": "Model"},
                        {"name": "Factory Hashrate", "id": "Capacity"},
                        {"name": "Wall Power Input", "id": "Power"},
                        {"name": "Unit Price ($ USD)", "id": "Price"},
                        {"name": "Net Profit ($/Day/Unit)", "id": "Net_Profit"},
                        {"name": "Estimated Payback (Days)", "id": "Payback"},
                        {"name": "Procurement Verdict Status", "id": "Status"}
                    ],
                    style_table={'overflowX': 'auto'},
                    style_cell={'textAlign': 'left', 'fontFamily': 'Segoe UI, Arial', 'padding': '12px', 'fontSize': '14px'},
                    style_header={'backgroundColor': '#f8f9fa', 'fontWeight': 'bold', 'color': '#2c3e50', 'borderBottom': '2px solid #e2e8f0'},
                    style_data_conditional=[
                        {
                            'if': {'column_id': 'Status', 'filter_query': '{Status} contains "🚀"'},
                            'backgroundColor': '#e8f5e9', 'color': '#2e7d32', 'fontWeight': 'bold'
                        },
                        {
                            'if': {'column_id': 'Status', 'filter_query': '{Status} contains "✅"'},
                            'backgroundColor': '#e1f5fe', 'color': '#0288d1', 'fontWeight': '500'
                        },
                        {
                            'if': {'column_id': 'Status', 'filter_query': '{Status} contains "❌"'},
                            'backgroundColor': '#ffebee', 'color': '#c62828'
                        }
                    ]
                )
            ], style={'backgroundColor': '#ffffff', 'padding': '20px', 'borderRadius': '8px', 'boxShadow': '0 4px 6px rgba(0,0,0,0.02)'})
            
        ], style={'width': '72%', 'padding': '25px', 'boxSizing': 'border-box'})
        
    ], style={'display': 'flex', 'flexDirection': 'row', 'backgroundColor': '#f1f2f6'})
])

# Master call-back logic processing pipeline
@app.callback(
    [Output('kpi-cards-grid', 'children'),
     Output('continuous-economic-curve', 'figure'),
     Output('realtime-procurement-table', 'data')],
    [Input('btc-price-slider', 'value'),
     Input('network-difficulty-slider', 'value'),
     Input('halving-era-dropdown', 'value'),
     Input('grid-electricity-cost', 'value'),
     Input('pool-settlement-fee', 'value')]
)
def run_integrated_financial_model(btc_price, difficulty_tier, era_factor, elec_cost, pool_fee):
    # 1. Base Bitcoin Math Calculations
    block_reward = 3.125 if era_factor == 'pre' else 1.5625
    difficulty_raw = difficulty_tier * 1_000_000_000_000 # Convert T to raw numerical integer
    
    raw_th_revenue_day = ((10**12 * 86400 * block_reward) / (difficulty_raw * 2**32)) * btc_price
    net_th_revenue_day = raw_th_revenue_day * (1 - pool_fee)
    
    max_efficiency_ceiling = (net_th_revenue_day / elec_cost) * (1000.0 / 24.0)
    procurement_target = max_efficiency_ceiling * 0.70
    
    # 2. Strategy Card Generation (Zone B)
    if max_efficiency_ceiling < 12.0:
        status_text, status_color, status_sub = "CRITICAL OBSOLESCENCE RISK", "#e74c3c", "Only unreleased next-gen hydro fleets viable."
    elif max_efficiency_ceiling < 17.0:
        status_text, status_color, status_sub = "HIGH RISK / TIGHT MARGINS", "#f39c12", "Requires premium sub-14 J/TH hardware assets."
    else:
        status_text, status_color, status_sub = "HEALTHY FLEET RUNWAY", "#2ecc71", "Standard commercial air-cooled rigs operational."

    kpi_elements = [
        html.Div([
            html.Div("MAX VIABLE EFFICIENCY", style={'fontSize': '11px', 'fontWeight': 'bold', 'color': '#7f8c8d'}),
            html.Div(f"{max_efficiency_ceiling:.2f} J/TH", style={'fontSize': '24px', 'fontWeight': 'bold', 'color': '#c0392b'}),
            html.Div("Absolute Breakeven Threshold", style={'fontSize': '11px', 'color': '#95a5a6', 'marginTop': '4px'})
        ], style={'backgroundColor': '#fff', 'padding': '20px', 'borderRadius': '6px', 'width': '31%', 'borderLeft': '5px solid #c0392b', 'boxShadow': '0 2px 5px rgba(0,0,0,0.04)'}),
        
        html.Div([
            html.Div("RECOMMENDED TARGET", style={'fontSize': '11px', 'fontWeight': 'bold', 'color': '#7f8c8d'}),
            html.Div(f"{procurement_target:.2f} J/TH", style={'fontSize': '24px', 'fontWeight': 'bold', 'color': '#27ae60'}),
            html.Div("Maintains a 30% OpEx Cushion", style={'fontSize': '11px', 'color': '#95a5a6', 'marginTop': '4px'})
        ], style={'backgroundColor': '#fff', 'padding': '20px', 'borderRadius': '6px', 'width': '31%', 'borderLeft': '5px solid #27ae60', 'boxShadow': '0 2px 5px rgba(0,0,0,0.04)'}),
        
        html.Div([
            html.Div("FLEET ENVIRONMENT STATUS", style={'fontSize': '11px', 'fontWeight': 'bold', 'color': '#7f8c8d'}),
            html.Div(status_text, style={'fontSize': '14px', 'fontWeight': 'bold', 'color': status_color, 'marginTop': '8px'}),
            html.Div(status_sub, style={'fontSize': '11px', 'color': '#95a5a6', 'marginTop': '4px'})
        ], style={'backgroundColor': '#fff', 'padding': '20px', 'borderRadius': '6px', 'width': '31%', 'borderLeft': f'5px solid {status_color}', 'boxShadow': '0 2px 5px rgba(0,0,0,0.04)'})
    ]

    # 3. Continuous Yield Calculation Matrix (Zone C - Fixed per PH/s Scale)
    x_spectrum = list(range(8, 33)) # 8 J/TH to 32 J/TH bounds
    ph_profits = []
    
    for x_val in x_spectrum:
        daily_power_cost_ph = (x_val / 1000.0) * 1000 * 24 * elec_cost
        daily_revenue_ph = net_th_revenue_day * 1000
        ph_profits.append(daily_revenue_ph - daily_power_cost_ph)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=x_spectrum, y=ph_profits, mode='lines+markers', name='Daily Profit per PH/s',
        line=dict(color='#2980b9', width=3), marker=dict(size=5)
    ))
    fig.add_vline(x=max_efficiency_ceiling, line_width=2, line_dash="dash", line_color="#c0392b")
    
    # Shade dynamic visual background boundaries
    valid_mid = min(max(max_efficiency_ceiling, 8), 32)
    fig.add_vrect(x0=8, x1=valid_mid, fillcolor="#2ecc71", opacity=0.07, line_width=0, layer="below")
    fig.add_vrect(x0=valid_mid, x1=32, fillcolor="#e74c3c", opacity=0.07, line_width=0, layer="below")

    fig.update_layout(
        title="Daily Operational Net Profit / Loss Curve (Per PH/s)",
        xaxis_title="ASIC Hardware Efficiency Profile (J/TH) — Lower is More Efficient",
        yaxis_title="Net Yield Balance ($ USD / Day)", 
        template="plotly_white",
        xaxis=dict(range=[8, 32], dtick=2), 
        yaxis=dict(zeroline=True, zerolinewidth=1.5, zerolinecolor='#7f8c8d'),
        margin=dict(l=40, r=40, t=50, b=40)
    )

    # 4. Hardware Catalog Table Mapping Processing Loop (Zone D)
    table_records = []
    for _, miner in df_hardware.iterrows():
        # Evaluate standard specific unit parameters
        miner_unit_rev = net_th_revenue_day * miner["Hashrate_TH"]
        miner_unit_power_kwh = (miner["Power_W"] / 1000.0) * 24.0
        miner_unit_cost = miner_unit_power_kwh * elec_cost
        miner_unit_profit = miner_unit_rev - miner_unit_cost
        
        # Calculate payback returns
        if miner_unit_profit > 0:
            payback_days = int(round(miner["Market_Price_USD"] / miner_unit_profit, 0))
            payback_display = f"{payback_days:,} Days"
            
            if miner["Efficiency_JTH"] <= procurement_target:
                verdict = "🚀 OPTIMAL CHOICE (Target Horizon)"
            else:
                verdict = "✅ VIABLE ASSET (Standard Yield)"
        else:
            payback_display = "Infinite (Operating at Loss)"
            verdict = "❌ UNSAFE OVERHEAD (Obsolete Fleet)"
            
        table_records.append({
            "Model": miner["Model"],
            "Capacity": f"{miner['Hashrate_TH']:.0f} TH/s",
            "Power": f"{miner['Power_W']:.0f} W",
            "Price": f"${miner['Market_Price_USD']:,}",
            "Net_Profit": f"${miner_profit_day: .2f}" if 'miner_profit_day' in locals() else f"${miner_unit_profit:.2f}",
            "Payback": payback_display,
            "Status": verdict
        })

    return kpi_elements, fig, table_records

if __name__ == '__main__':
    # Initialized on an isolated local network port for clean notebook compilation
    app.run(debug=True, jupyter_mode="inline", port=8068)

# Bitcoin Infrastructure Efficiency & Payback Engine

An enterprise-grade financial simulation platform built to model Bitcoin mining economics through the lens of thermodynamic asset valuation. Instead of evaluating hardware profitability via fixed manufacturer brand labels, this framework reverse-engineers global network parameters to treat **ASIC Efficiency ($J/\text{TH}$)** as the definitive dependent variable driving facility survival and procurement strategies.

## 🛠️ The Core Mathematical Logic

This platform mathematically isolates the maximum power tariff overhead a facility can withstand before hardware operating costs entirely consume mining yields.

### 1. Maximum Viable Efficiency Ceiling ($E_{\text{max}}$)
The point at which daily revenue per Terahash exactly matches the electrical cost to generate it ($0.00 breakeven baseline):

$$E_{\text{max}} = \left( \frac{\left( \frac{10^{12} \times 86400 \times BR}{Difficulty \times 2^{32}} \right) \times P_{\text{BTC}} \times (1 - Fee)}{R_{\text{elec}}} \right) \times \left( \frac{1000}{24} \right)$$

### 2. CapEx Payback Horizon Matrix
Calculates the absolute capital depreciation return schedules by checking live operational unit net margins against dynamic wholesale equipment market price layers:

$$\text{Estimated Payback (Days)} = \frac{\text{Wholesale Equipment Unit Price (\$)}}{\text{Daily Unit Revenue (\$/Day)} - \text{Daily Unit OpEx Cost (\$/Day)}}$$

---

## 📊 Application Visual Architecture

The dashboard implements an interactive **Sidebar Layout** divided into four major analytical zones:

*   **Zone A (Input Console):** Responsive sliders enabling real-time stress testing of variables, including Bitcoin spot price, global network difficulty epochs, local electricity tariffs ($/kWh), and pool fees.
*   **Zone B (Executive Readout Cards):** High-contrast KPI displays presenting the calculated Maximum Viable Efficiency, the Recommended Sourcing Target (incorporating a safe 30% OpEx margin cushion), and an automated structural risk verdict.
*   **Zone C (Continuous Yield Curve Plot):** Generates a continuous thermodynamic efficiency spectrum curve scaled to **Petahashes per second (PH/s)**, explicitly shading the boundaries between profitable territory and the grid obsolescence zone.
*   **Zone D (Wholesale Sourcing Grid):** A tabular procurement lookup matrix that maps live market miner models against the calculated economic threshold, dynamically computing single-unit margins and exact capital return (ROI) windows.

---

## 🚀 Quick Start Guide

### Prerequisites
Ensure your local Python environment or Jupyter Notebook instance has the required visualization dependencies installed:

```bash
pip install dash plotly pandas